In [ ]:
from bokeh.server.server import Server
from bokeh.application import Application
from bokeh.application.handlers.function import FunctionHandler

import numpy as np
from collections import OrderedDict
from pprint import pprint
from bokeh.io import show, output_file, output_notebook
from bokeh.layouts import row, column
from bokeh.palettes import Viridis6,Category20
from bokeh.plotting import figure
from bokeh.models import CustomJS, Legend, Toggle, CheckboxGroup
from bokeh.layouts import gridplot, widgetbox
from bokeh.colors import RGB
import matplotlib.colors as colors
import matplotlib.cm as cmx
import pylab



### Jupyter notebook display settings
The next cell is being used to remove padding from the jupyter notebook rendering to increase the size of the cells

In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>"
    + "#notebook { padding-top:0px !important; } " 
    + ".container { width:100% !important; } "
    + ".end_space { min-height:0px !important; } "
    + "</style>"))

### Demonstration of Checkbox batch selection and legend toggling
The following example demonstrates how multiple selections can be done using the same check box.

It also shows how the legends can be turned on and off with the selection on a checkbox

#### New Additions
- Added automatic legend object determination
- Automatic addition of legends to plot

In [ ]:
#to allow inline plotting
#output_file('doc.html')
#output_notebook()
mycmap='coolwarm'
cNorm = colors.Normalize(vmin=0,vmax=64-1)
mymap = cm = pylab.get_cmap(mycmap)
scalarMap = cmx.ScalarMappable(norm=cNorm,cmap=mymap)

PLOT_HEIGHT = 700
PLOT_WIDTH = 1000
#output_file('here.html')
output_notebook()
def color_denormalize(ycol):
    '''
        converting rgb values from 0 to 255

    '''
    ycol=np.array(ycol)
    ycol=np.array(ycol*255,dtype=int)
    ycol=ycol.tolist()
    ycol=tuple(ycol)[:-1]

    return ycol


#creating the figure
p = figure(plot_width=PLOT_WIDTH, plot_height=PLOT_HEIGHT)
b = figure(plot_width=PLOT_WIDTH, plot_height=PLOT_HEIGHT)
props = dict(line_width=4, line_alpha=0.7)

#creating the data
x = np.linspace(0, 4 * np.pi, 100)

#lists to store generated legend names and glyph states.
#glyph states stored because a single glyph element, l0, is shared throught the iteration
gen_legends = []
glyph_states_l0=[]
glyph_states_l1=[]

#for i in range(3):
for i in range (64):
    col = scalarMap.to_rgba(float(i))
    col = color_denormalize(col)

    #creating the legend
    l= "%s"%str(i)

    l0 = p.line(x, i*np.sin(x), color=col, **props)
    #l1 = b.line(x, i*np.cos(x), color=col, **props)

    #Setting single plot to be visibe at the beginning
    if i>0:
        l0.visible=False
        

    glyph_states_l0.append( (l,[l0]) )
    #glyph_states_l1.append( (l,[l1]) )
    gen_legends.append(l)

    
    
#determining the number of legend objects required to be created
num_legend_objs =  int( np.ceil( len(gen_legends) / float(16) ) )



#Automating creating batches of 16 each unless otherwise
#Looks like
# batch_0 : glyph_states_l0[:16]
#equivalent to
# batch_0 = glyph_states_l0[:16]
batches = {}
j = 0
for i in range(num_legend_objs):
    #incase the number is not a multiple of 16
    if i == num_legend_objs:
        batches["batch_%s"%str(i)] = glyph_states_l0[j:]
    else:
        batches["batch_%s"%str(i)] = glyph_states_l0[j:j+16]
    j+= 16


    

#creating legend objects using items from the previous batches dictionary
#Legend objects allow their positioning outside the main plot
#resulting ordered dictionary looks like; 
# leg_0 : Legend(items=batch_0, location='top_right', click_policy='hide')
#equivalent to
# leg_0 = Legend(items=batch_0, location='top_right', click_policy='hide')


legend_objs = OrderedDict()
for i in range(num_legend_objs):
    legend_objs['leg_%s'%str(i)] = Legend(items=batches["batch_%s"%str(i)], location='top_right', click_policy='hide')


    

ant_labels = ["Ant 00 - 15", "Ant 16 - 31", "Ant 32 - 47","Ant 48 - 63"]

#select all antennas button
sel_all = Toggle(label='Select All Antennas', button_type='success')
batch_sel = CheckboxGroup(labels=ant_labels, active=[])


#setting the callback for the button
#Bokeh models can be specified as arguments for manipulation within the js code
#legends = legend_objs.keys()
sel_all.callback = CustomJS( args = dict(glyp = glyph_states_l0, glyp2 = glyph_states_l1, batch_sel = batch_sel, legend_objs = legend_objs.values() ), code = '''
    var i, bins;

    //if toggle button active
    if (this.active==false)
        {
            cb_obj.label='Select all Antennas';

            //
            for(i=0; i<glyp.length; i++){
                glyp[i][1][0].visible = false;
                //glyp2[i][1][0].visible = false;
            }
            batch_sel.active = [];
            
            for(j=0; j<legend_objs.length; j++){
                legend_objs[j].visible =false;
            }
        }
    else{
            cb_obj.label='Deselect all Antennas';
            for(i=0; i<glyp.length; i++){
                glyp[i][1][0].visible = true;
                //glyp2[i][1][0].visible = true;
            }
            batch_sel.active = [0,1,2,3];
            
            for(j=0; j<legend_objs.length; j++){
                legend_objs[j].visible = true;
            }
        }
''')
 
batch_sel.callback = CustomJS.from_coffeescript(args = dict( glyp = glyph_states_l0, legend_objs = legend_objs.values() ), code = '''
m = 0
while m<legend_objs.length
    legend_objs[m].visible = false
    m++
    
if 0 in cb_obj.active
    legend_objs[0].visible = true
    i = 0
    until i>16
        glyp[i][1][0].visible = true
        i++
else
    legend_objs[0].visible = false
    i = 0
    until i>16
        glyp[i][1][0].visible = false
        i++
if 1 in cb_obj.active
    legend_objs[1].visible = true
    i = 16
    until i>31
        glyp[i][1][0].visible = true
        i++
else
    legend_objs[1].visible = false
    i = 16
    until i>31
        glyp[i][1][0].visible = false
        i++
       
      
if 2 in cb_obj.active
    legend_objs[2].visible = true
    i = 32
    until i>48
        glyp[i][1][0].visible = true
        i++
else
    legend_objs[2].visible = false
    i = 32
    until i>48
        glyp[i][1][0].visible = false
        i++
        
if 3 in cb_obj.active
    legend_objs[3].visible = true
    i = 48
    until i>65
        glyp[i][1][0].visible = true
        i++
else
    legend_objs[3].visible = false
    i = 48
    until i>65
        glyp[i][1][0].visible = false
        i++

    ''')



p.title.text="Dummy B vs Dummy A"
b.title.text="Dummy C vs Dummy D"

#to add the legends to the layout in an ordered manner
for key in sorted( legend_objs.keys()):
    p.add_layout(legend_objs[key], "right")

#p.add_layout(leg, 'right')
#p.add_layout(lega, 'right')
#p.add_layout(legb, 'right')
#p.add_layout(legc, 'right')

#switch off legends to expose more of the plot
legs_off = Toggle(label ="on/off legends",active=False, button_type="success")
legs_off.callback = CustomJS(args = dict(legend_objs = legend_objs.values() ), code = """
    if (cb_obj.active){
        for (var i=0; i<legend_objs.length; i++){
            legend_objs[i].visible = false;
        }
            
        
    }
    else{
         for (var i=0; i<legend_objs.length; i++){
            legend_objs[i].visible = true;
        }
    }

""")
widgets = widgetbox(sel_all, batch_sel, legs_off)
lay= column(widgets,p)
layout = column(lay, b)
show(lay)

## Running the script from terminal

The next cell runs and external script giving output ack on the cell.
REquires a deeper understanding of how the hook works for notebook output.

In [ ]:
%run /home/lexya/Documents/msc_project/gain_tables/plot_utils/plot_delay_K_ed.py  /home/lexya/Documents/msc_project/gain_tables/plot_utils/1491291289.K0/

### Running the scripts as library functions

In [ ]:
import nbgains

In [ ]:
nbgains.plot_G_table('/home/lexya/Documents/msc_project/gain_tables/plot_utils/1491291289.G0/', doplot="ap", corr=1)


In [ ]:
nbgains.plot_B_table('/home/lexya/Documents/msc_project/gain_tables/plot_utils/1491291289.B0/')

In [ ]:
nbgains.plot_K_table('/home/lexya/Documents/msc_project/gain_tables/plot_utils/1491291289.K0/', field=1)